# Know Your Village

Explore population, services, livestock, village survey records and nearby micro-watersheds.

Run the cells in order. Each step uses data from the previous cells. You can edit the place, identifier, columns and chart settings as you go.


## Set up Python

Run these two collapsed cells once. They load the libraries and starting location. Expand them to see or change the setup.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import ast
import json
from getpass import getpass
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink


In [ ]:
SCOPE = json.loads("{\"state\": \"Bihar\", \"district\": \"Nalanda\", \"tehsil\": \"Hilsa\"}")
GEOSERVER = 'https://geoserver.core-stack.org:8443/geoserver/'
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


## Choose the tehsil

The starting place is Hilsa, Nalanda, Bihar. A notebook downloaded from GeoLibre uses the selected tehsil. Change these names to read another place.


In [ ]:
state = SCOPE["state"].lower().replace(" ", "_")
district = SCOPE["district"].lower().replace(" ", "_")
tehsil = SCOPE["tehsil"].lower().replace(" ", "_")
place = {"state": state, "district": district, "tehsil": tehsil}
place


## The layers we will use

Each link reads a vector layer as GeoJSON. GeoJSON contains a feature list; each feature has a shape and its data fields.


In [ ]:
layer_urls = {
    "facilities": f"{GEOSERVER}facilities_proximity/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=facilities_proximity:facilities_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
    "population": f"{GEOSERVER}panchayat_boundaries/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=panchayat_boundaries:{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
    "livestock": f"{GEOSERVER}livestocks/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=livestocks:livestocks_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
    "survey": f"{GEOSERVER}antyodaya_2020/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=antyodaya_2020:antyodaya20_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
    "mws": f"{GEOSERVER}mws/ows?service=WFS&version=2.0.0&request=GetFeature&typeNames=mws:mws_{district}_{tehsil}&outputFormat=application/json&srsName=EPSG:4326",
}
pd.DataFrame(layer_urls.items(), columns=["Layer", "GeoJSON URL"])


## Read village services

The facilities layer contains village names, identifiers and distances in kilometres.


In [ ]:
facilities_response = requests.get(layer_urls["facilities"], timeout=90)
facilities_response.raise_for_status()
facilities_geojson = facilities_response.json()
facilities = gpd.GeoDataFrame.from_features(facilities_geojson["features"], crs="EPSG:4326")
facilities.drop(columns="geometry").head()


## Choose a village

Choose from the village name and identifier list. Replace `village_id` to explore another village.


In [ ]:
villages = facilities[["village_name", "village_id"]].dropna(subset=["village_id"]).sort_values("village_id")
display(villages)
village_id = int(villages.iloc[0]["village_id"])
village_services = facilities.loc[facilities["village_id"] == village_id].iloc[0]
village_id


## Read population and literacy

The census layer uses `vill_ID` for the village identifier. Population and literacy fields below are counts of people.


In [ ]:
population_response = requests.get(layer_urls["population"], timeout=90)
population_response.raise_for_status()
population_geojson = population_response.json()
population = gpd.GeoDataFrame.from_features(population_geojson["features"], crs="EPSG:4326")
population.drop(columns="geometry").head()


## Read the field descriptions

STAC records describe the published fields. This table selects the fields used below and keeps their original descriptions.


In [ ]:
item_name = f"{state}_{district}_{tehsil}_admin_boundaries_vector"
item_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/{item_name}/{item_name}.json")
item_response = requests.get(item_url, timeout=90)
item_response.raise_for_status()
item = item_response.json()
field_notes = pd.DataFrame(item["properties"]["table:columns"])
field_notes.loc[field_notes["name"].isin(['vill_ID', 'TOT_P', 'P_SC', 'P_ST', 'P_LIT']), ["name", "type", "description"]]


## Population and literacy

Read the total population, Scheduled Caste and Scheduled Tribe counts, and recorded literacy counts.


In [ ]:
selected_village = population.loc[population["vill_ID"] == village_id]
population_fields = {"TOT_P": "Population", "TOT_M": "Male population", "TOT_F": "Female population",
                     "P_SC": "Scheduled Caste population", "P_ST": "Scheduled Tribe population",
                     "P_LIT": "Literate people", "M_LIT": "Literate men", "F_LIT": "Literate women", "P_ILL": "Illiterate people"}
selected_village[list(population_fields)].rename(columns=population_fields).T


## Distances to services

These are the recorded service distances in kilometres.


In [ ]:
service_fields = {'l2_essential_education_distance_km': 'Essential education',
 'l2_higher_education_distance_km': 'Higher education',
 'l2_essential_health_distance_km': 'Essential health',
 'l2_advanced_health_distance_km': 'Advanced health',
 'l2_essential_services_distance_km': 'Essential services',
 'l2_financial_inclusion_distance_km': 'Financial services',
 'l2_apmc_access_distance_km': 'Agricultural markets',
 'l2_post_harvest_distance_km': 'Post-harvest services',
 'l2_cooperative_distance_km': 'Cooperatives',
 'l2_livestock_distance_km': 'Livestock services',
 'l2_agri_support_infra_distance_km': 'Agricultural support infrastructure'}
service_distances = village_services[list(service_fields)].rename(index=service_fields).astype(float)
service_distances.to_frame("Distance (km)")


## Compare service distances

Longer bars indicate services farther from the village.


In [ ]:
service_distances.plot.barh(figsize=(9, 5), xlabel="Distance (km)")
plt.tight_layout()
plt.show()


## Read livestock counts

The livestock layer uses the same `village_id`.


In [ ]:
livestock_response = requests.get(layer_urls["livestock"], timeout=90)
livestock_response.raise_for_status()
livestock_geojson = livestock_response.json()
livestock = gpd.GeoDataFrame.from_features(livestock_geojson["features"], crs="EPSG:4326")
livestock.drop(columns="geometry").head()


## Livestock in the village

Compare the recorded counts for the main animal groups.


In [ ]:
village_livestock = livestock.loc[livestock["village_id"] == village_id]
livestock_fields = {"all_livestock_total": "All livestock", "cattle_total": "Cattle", "buffalo_total": "Buffalo",
                    "sheep_total": "Sheep", "goat_total": "Goats", "pig_total": "Pigs"}
village_livestock[list(livestock_fields)].rename(columns=livestock_fields).T


## Read Mission Antyodaya records

Mission Antyodaya records describe village facilities and livelihoods. The following tables pair category values with the original survey fields. Survey answers retain their published values.


In [ ]:
survey_response = requests.get(layer_urls["survey"], timeout=90)
survey_response.raise_for_status()
survey_geojson = survey_response.json()
survey = gpd.GeoDataFrame.from_features(survey_geojson["features"], crs="EPSG:4326")
survey.drop(columns="geometry").head()


## Select the village survey

Use the same village identifier to read its survey record.


In [ ]:
village_survey = survey.loc[survey["village_id"] == village_id].iloc[0]
village_survey[["village_name", "village_id"]]


## Survey field reference

This collapsed reference pairs each survey group with its original fields and question labels. Run it once, then choose a group below.


In [ ]:
survey_groups = {'road_connectivity': {'is_village_connected_to_all_weather_road': 'All-weather road connection',
                       'availability_of_internal_pucca_road': 'Internal pucca road quality',
                       'availability_of_public_transport': 'Public transport availability',
                       'availability_of_railway_station': 'Railway station availability'},
 'energy_access': {'availablility_hours_of_domestic_electricity': 'Domestic electricity supply '
                                                                  '(hours/day)',
                   'availability_of_elect_supply_to_msme': 'Electricity supply to MSME units',
                   'total_hhd': 'Total number of households',
                   'total_hhd_with_clean_energy': 'HHs using clean energy (LPG / Biogas)'},
 'housing_quality': {'total_hhd': 'Total number of households',
                     'total_hhd_with_kuccha_wall_kuccha_roof': 'HHs with kuccha wall & kuccha roof',
                     'total_hhd_got_benefit_under_state_housing_scheme': 'State housing scheme '
                                                                         'beneficiaries',
                     'total_hhd_have_got_pmay_house': 'PMAY houses (completed / sanctioned)',
                     'total_hhd_in_pmay_permanent_wait_list': 'PMAY permanent waitlist households',
                     'total_hhd_availing_pmuy_benefits': 'PMUY (Ujjwala Yojana) beneficiaries'},
 'maternal_child_health': {'availability_of_mother_child_health_facilities': 'Availability of '
                                                                             'Mother and Child '
                                                                             'Health facilities',
                           'is_aanganwadi_centre_available': 'Availability of Aanganwadi Centre',
                           'is_early_childhood_edu_provided_in_anganwadi': 'Is Early Childhood '
                                                                           'Education provided in '
                                                                           'the Anganwadi',
                           'total_childs_aged_0_to_3_years': 'Total no of children in the age '
                                                             'group of 0-3 years',
                           'total_childs_aged_0_to_3_years_reg_under_aanganwadi': 'Total no of '
                                                                                  'children aged '
                                                                                  '0-3 years '
                                                                                  'registered in '
                                                                                  'Aanganwadi',
                           'total_no_of_pregnant_women': 'Total number of Pregnant women',
                           'total_no_of_pregnant_women_receiving_services_under_icds': 'No of '
                                                                                       'pregnant '
                                                                                       'women '
                                                                                       'receiving '
                                                                                       'services '
                                                                                       'under ICDS',
                           'total_no_of_lactating_mothers': 'Total number of lactating mothers',
                           'total_anemic_pregnant_women': 'No. of Anaemic Pregnant Women',
                           'total_childs_aged_0_to_3_years_immunized': 'No of children aged 0-3 '
                                                                       'years immunized',
                           'total_no_of_newly_born_children': 'Total number of newly born children '
                                                              'during the year',
                           'total_no_of_newly_born_underweight_children': 'No of newly born '
                                                                          'children underweight',
                           'gp_total_no_of_beneficiaries_receiving_benefits_under_pmjay': 'No. of '
                                                                                          'beneficiaries '
                                                                                          'receiving '
                                                                                          'benefits '
                                                                                          'under '
                                                                                          'PMJAY',
                           'gp_total_no_of_eligible_beneficiaries_under_pmjay': 'Total no. of '
                                                                                'eligible '
                                                                                'beneficiaries '
                                                                                'under PMJAY',
                           'total_hhd_registered_under_pmjay': 'No. of Households registered under '
                                                               'PMJAY/State Health Insurance',
                           'total_no_of_beneficiaries_receiving_benefits_under_pmmvy': 'No of '
                                                                                       'beneficiaries '
                                                                                       'receiving '
                                                                                       'benefits '
                                                                                       'under '
                                                                                       'PMMVY',
                           'total_no_of_eligible_beneficiaries_under_pmmvy': 'Total no of eligible '
                                                                             'beneficiaries under '
                                                                             'PMMVY'},
 'water_sanitation': {'availability_of_piped_tap_water': 'Availability of Piped tap water '
                                                         '(Coverage)',
                      'total_hhd': 'Total number of households',
                      'total_hhd_having_piped_water_connection': 'No of households having piped '
                                                                 'water connection',
                      'total_hhd_not_having_sanitary_latrines': 'No of households not having '
                                                                'sanitary latrines',
                      'availability_of_drainage_system': 'Availability of drainage facilities',
                      'is_community_waste_disposal_system': 'Community waste disposal system',
                      'is_community_biogas_waste_recycle_for_production': 'Community bio gas or '
                                                                          'recycle of waste'},
 'financial_inclusion': {'is_bank_available': 'Availability of banks',
                         'is_atm_available': 'Availability of ATM',
                         'is_bank_buss_correspondent_with_internet': 'Availability of Business '
                                                                     'Correspondent with internet '
                                                                     'connectivity',
                         'total_shg': 'Number of Self Help Groups (SHGs)',
                         'total_shg_accessed_bank_loans': 'No of SHGs which accessed bank loans',
                         'total_hhd': 'Total number of households',
                         'total_hhd_availing_pmjdy_bank_ac': 'Number of households having Jan-Dhan '
                                                             'bank account'},
 'social_protection': {'gp_total_hhd_eligible_under_nfsa': 'Total number of eligible households '
                                                           'under NFSA',
                       'gp_total_hhd_receiving_food_grains_from_fps': 'Total no of households '
                                                                      'receiving food grains from '
                                                                      'Fair Price Shops',
                       'total_hhd': 'Total households',
                       'total_hhd_having_bpl_cards': 'Number of Households having BPL ration cards',
                       'total_hhd_availing_pension_under_nsap': 'Number of Households getting '
                                                                'pensions under NSAP'},
 'institutionalization': {'total_hhd': 'Total number of households',
                          'total_hhd_mobilized_into_shg': 'Number of households mobilized into '
                                                          'SHGs',
                          'total_no_of_shg_promoted': 'Number of SHGs federated into Village '
                                                      'Organisations',
                          'total_shg': 'Number of Self Help Groups (SHGs)',
                          'total_hhd_mobilized_into_pg': 'Number of households mobilized into '
                                                         'Producer Groups',
                          'availability_of_fpos_pacs': 'Availability of Farmers Collective (Farmer '
                                                       'Producer Organizations (FPOs)/Primary '
                                                       'Agricultural Credit Societies (PACS))'},
 'civic_infrastructure': {'availability_of_panchayat_bhawan': 'Availability of Panchayat Bhawan',
                          'is_post_office_available': 'Availability of Post office/Sub-Post office',
                          'total_no_of_elected_representatives': 'Total no of elected '
                                                                 'representatives',
                          'total_no_of_elect_rep_undergone_training_under_rgsa': 'No of elected '
                                                                                 'representatives '
                                                                                 'undergone '
                                                                                 'refresher '
                                                                                 'training under '
                                                                                 'RGSA',
                          'total_no_of_elect_rep_oriented_under_rgsa': 'No of elected '
                                                                       'representatives oriented '
                                                                       'under RGSA',
                          'availability_of_public_information_board': 'Availability of Public '
                                                                      'Information Board under '
                                                                      "People's Plan Campaign",
                          'availability_of_public_library': 'Availability of Public Library'},
 'livelihoods_employment': {'total_hhd': 'Total number of households',
                            'total_hhd_engaged_in_farm_activities': 'Number of households engaged '
                                                                    'majorly in farm activities'},
 'livelihoods_forest_resources': {'availability_of_community_forest': 'Availability of Community '
                                                                      'Forest',
                                  'availability_of_minor_forest_production': 'Availability of '
                                                                             'minor forest '
                                                                             'production',
                                  'total_hhd': 'Total number of households',
                                  'total_hhd_source_of_minor_forest_production': 'Number of '
                                                                                 'Households where '
                                                                                 'only source of '
                                                                                 'livelihood is '
                                                                                 'minor forest '
                                                                                 'production'},
 'livelihoods_fisheries': {'availability_of_aquaculture_ext_facility': 'Extension facilities for '
                                                                       'Aquaculture',
                           'availability_of_fish_community_ponds': 'Community Ponds Used for '
                                                                   'Fisheries',
                           'availability_of_fish_farming': 'Pisciculture - InLand Fishery/Coastal '
                                                           'Fishery'},
 'livelihoods_alternative_farming': {'is_bee_farming': 'Bee Keeping',
                                     'is_sericulture': 'Sericulture (Silk Production)'},
 'livelihoods_cottage_traditional_industry': {'availability_of_cottage_small_scale_units': 'Availability '
                                                                                           'of '
                                                                                           'cottage '
                                                                                           'and '
                                                                                           'small '
                                                                                           'scale '
                                                                                           'units',
                                              'total_hhd': 'Total number of households',
                                              'total_hhd_engaged_cottage_small_scale_units': 'Number '
                                                                                             'of '
                                                                                             'Households '
                                                                                             'engaged '
                                                                                             'in '
                                                                                             'cottage/small '
                                                                                             'scale '
                                                                                             'units',
                                              'is_handloom': 'Handloom',
                                              'is_handicrafts': 'Handicrafts'},
 'livelihoods_common_resources': {'is_common_pastures_available': 'Common pastures as per revenue '
                                                                  'records'},
 'livestock_veterinary': {'availability_of_livestock_extension_services': 'Availability of '
                                                                          'Livestock Extension '
                                                                          'services',
                          'is_veterinary_hospital_available': 'Availability of Veterinary Clinic '
                                                              'or Hospital',
                          'availability_of_goatary_dev_project': 'Project supporting Goatary '
                                                                 'Development',
                          'availability_of_pigery_development': 'Project supporting Piggery '
                                                                'Development',
                          'availability_of_poultry_dev_project': 'Project supporting Poultry '
                                                                 'Development',
                          'availability_of_milk_routes': 'Availability of Milk Collection '
                                                         'Centre/Milk routes/Chilling Centres'},
 'agriculture_land_cultivation': {'area_irrigated_in_hac': 'Total area irrigated (ha)',
                                  'net_sown_area_in_hac': 'Net sown Area (ha)',
                                  'net_sown_area_kharif_in_hac': 'Net sown Area during Kharif '
                                                                 'season (ha)',
                                  'net_sown_area_other_in_hac': 'Net sown Area during other '
                                                                'seasons (ha)',
                                  'net_sown_area_rabi_in_hac': 'Net sown Area during Rabi season '
                                                               '(ha)',
                                  'total_cultivable_area_in_hac': 'Total Cultivable Area (ha)'},
 'agriculture_irrigation_watershed': {'availability_of_major_source_of_irrigation': 'Main Source '
                                                                                    'of irrigation',
                                      'availability_of_rain_harvest_system': 'Availability of '
                                                                             'Community Rain Water '
                                                                             'Harvesting '
                                                                             'System/Pond/Dam/Check '
                                                                             'Dam',
                                      'availability_of_watershed_dev_project': 'Whether village is '
                                                                               'part of Watershed '
                                                                               'Development '
                                                                               'Project',
                                      'total_approved_labour_budget_for_year': 'Total approved '
                                                                               'Labour Budget for '
                                                                               'the year (₹)',
                                      'total_expenditure_approved_under_nrm_labour_budget_during_yr': 'Total '
                                                                                                      'expenditure '
                                                                                                      'approved '
                                                                                                      'under '
                                                                                                      'NRM '
                                                                                                      'in '
                                                                                                      'the '
                                                                                                      'Labour '
                                                                                                      'Budget '
                                                                                                      '(₹)',
                                      'no_of_farmers_using_drip_sprinkler': 'Number of farmers '
                                                                            'using drip/sprinkler '
                                                                            'irrigation',
                                      'total_no_of_farmers': 'Total no of farmers'},
 'agriculture_support_services': {'is_fertilizer_shop_available': 'Availability of fertilizer shop',
                                  'is_govt_seed_centre_available': 'Availability of government '
                                                                   'seed centres',
                                  'is_soil_testing_centre_available': 'Availability of soil '
                                                                      'testing centres',
                                  'total_no_of_farmers': 'Total no of farmers',
                                  'total_no_of_farmers_received_benefit_under_pmfby': 'No of '
                                                                                      'farmers '
                                                                                      'received '
                                                                                      'benefits '
                                                                                      'under PMFBY',
                                  'total_no_of_farmers_registered_under_pmkpy': 'Total number of '
                                                                                'farmers '
                                                                                'registered under '
                                                                                'PM Kisan Pension '
                                                                                'Yojana',
                                  'total_no_of_farmers_add_fert_in_soil_as_per_report': 'Number of '
                                                                                        'farmers '
                                                                                        'received '
                                                                                        'the soil '
                                                                                        'testing '
                                                                                        'report'},
 'agricultural_markets': {'availability_of_market': 'Availability of markets',
                          'availability_of_food_storage_warehouse': 'Availability of warehouse for '
                                                                    'Food Grain Storage'},
 'agriculture_organic_farming': {'total_no_farmers_adopted_organic_farming': 'No of farmers '
                                                                             'adopted organic '
                                                                             'farming',
                                 'total_no_of_farmers': 'Total no of farmers'}}


## Choose a survey group

The list shows the available groups and their category values. Change `survey_group` to another name in the list, such as `energy_access`, `agriculture_land_cultivation` or `agricultural_markets`, then rerun the next cells.


In [ ]:
group_list = pd.DataFrame({"Group": list(survey_groups),
    "Category value": [village_survey[g + "_cat_value"] for g in survey_groups]})
display(group_list)
survey_group = "road_connectivity"


## Read the category and original answers

This table keeps the exact source field, question label and answer together. The category summarises the group; the survey answers describe the recorded details.


In [ ]:
fields = {survey_group + "_cat_value": "Category value",
          survey_group + "_cat_cluster": "Category group", **survey_groups[survey_group]}
survey_answers = pd.DataFrame({"Field": list(fields), "Question": list(fields.values()),
    "Recorded answer": village_survey.reindex(list(fields)).values})
with pd.option_context("display.max_colwidth", None):
    display(survey_answers)


## Compare category values

The chart summarises the published categories. Use the group selector above to inspect their original survey answers.


In [ ]:
group_list.set_index("Group")["Category value"].plot.barh(figsize=(10, 8), xlabel="Category value")
plt.tight_layout()
plt.show()


## Explore agricultural services

Choose `agriculture_land_cultivation`, `agriculture_support_services` or `agricultural_markets` above to read their survey answers. Here are the service distances from the facilities layer.


In [ ]:
service_distances.loc[["Agricultural markets", "Post-harvest services", "Agricultural support infrastructure"]].to_frame("Distance (km)")


## Bring agricultural services and survey data together

Join on the village identifier. Then change `agriculture_columns` to inspect other fields listed in the survey reference or facilities table. Multiple source matches remain separate rows.


In [ ]:
agriculture_columns = ["village_id", "agriculture_land_cultivation_cat_value", "agricultural_markets_cat_value"]
market_columns = ["village_id", "l2_apmc_access_distance_km", "l2_agri_support_infra_distance_km"]
agriculture = survey.loc[survey["village_id"] == village_id, agriculture_columns].merge(
    facilities.loc[facilities["village_id"] == village_id, market_columns], on="village_id", how="left")
agriculture


## Read micro-watersheds

Use the village boundary to find intersecting MWS. An intersection does not allocate village population or livestock to an MWS.


In [ ]:
mws_response = requests.get(layer_urls["mws"], timeout=90)
mws_response.raise_for_status()
mws_geojson = mws_response.json()
mws = gpd.GeoDataFrame.from_features(mws_geojson["features"], crs="EPSG:4326")
mws.drop(columns="geometry").head()


## Micro-watersheds linked to the village

List MWS whose geometry intersects the selected village boundary.


In [ ]:
village_boundary = selected_village.geometry.union_all()
mws.loc[mws.intersects(village_boundary), ["uid", "area_in_ha"]]


## Connect to the CoRE Stack API

Read endpoint specifications at [api-doc.core-stack.org](https://api-doc.core-stack.org). The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains how to register, generate an API key and use it. The key goes in the `X-API-Key` header. This cell reads `CORE_STACK_API_KEY` from your environment, or asks for it without showing it. The key is not written into the notebook. After each API request, run the collapsed parsing cell: it keeps the raw response text and reads it with `json.loads`, which accepts `NaN` as a missing numeric value. Expand the cell to inspect the code.


In [ ]:
from inspect import isawaitable

api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key

api_headers = {"X-API-Key": str(api_key).strip()}


## Read village boundaries from the API

`get_village_geometries` returns GeoJSON boundaries with `vill_ID` and `vill_name`. Read the tehsil once, then select the village used above.


In [ ]:
response = requests.get(API_URL + "get_village_geometries/", params=place, headers=api_headers, timeout=90)
response.raise_for_status()


In [ ]:
raw_api_data_string = response.text
api_payload = json.loads(raw_api_data_string.lstrip("\ufeff"))


In [ ]:
api_villages = gpd.GeoDataFrame.from_features(api_payload["features"], crs="EPSG:4326")
display(api_villages[["vill_ID", "vill_name"]])
api_village = api_villages.loc[api_villages["vill_ID"] == village_id]
api_village.plot(figsize=(6, 6), edgecolor="black", color="#d7e9f5")
plt.show()


## Look up a place from coordinates

Take a point inside the village and ask `get_admin_details_by_latlon` for its administrative names. You can replace the latitude and longitude with another location.


In [ ]:
point = api_village.geometry.iloc[0].representative_point()
coordinates = {"latitude": point.y, "longitude": point.x}
response = requests.get(API_URL + "get_admin_details_by_latlon/", params=coordinates, headers=api_headers, timeout=90)
response.raise_for_status()


In [ ]:
raw_api_data_string = response.text
api_payload = json.loads(raw_api_data_string.lstrip("\ufeff"))


In [ ]:
api_payload


## Read the tehsil tables from the API

`get_tehsil_data` returns a dictionary of tables for the same place. Here we make the request once and reuse the returned tables below.


In [ ]:
api_response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
api_response.raise_for_status()


In [ ]:
raw_api_data_string = api_response.text
api_payload = json.loads(raw_api_data_string.lstrip("\ufeff"))


In [ ]:
api_data = api_payload
pd.DataFrame({"Table": api_data.keys(), "Rows": [len(rows) for rows in api_data.values()]})


## Read village records from the API

Choose `social_economic_indicator`, `facilities_proximity`, `livestock` or `antyodaya`. The same village identifier selects the record for each table.


In [ ]:
table_name = 'social_economic_indicator'
api_village_table = pd.DataFrame(api_data[table_name])
api_village_table.loc[api_village_table['village_id'] == village_id].T
